# Parcial #1 — Análisis de ventas de un restaurante de comida rápida

**Asignatura:** Tópicos Especiales II  |  **Facilitador:** Rodrigo Yángüez

**Integrantes del grupo:** (completar nombres)

**Dataset:** Balaji Fast Food Sales (1.000 ventas, abril 2022 a marzo 2023)

**Contenido**
1. Carga e inspección inicial
2. Limpieza de datos (con justificación de cada decisión)
3. Reportes obligatorios (4)
4. Reportes adicionales (2) con pregunta de negocio
5. Indicadores clave (2 KPI)
6. Hallazgos para el dueño
7. Consideraciones finales

## 1. Carga e inspección inicial

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

ARCHIVO = "Balaji Fast Food Sales (1).csv"

# En Google Colab: si el archivo no está en el entorno, se abre el selector para subirlo (descargado de Moodle).
if not os.path.exists(ARCHIVO):
    from google.colab import files
    subido = files.upload()
    ARCHIVO = list(subido.keys())[0]

df_original = pd.read_csv(ARCHIVO)   # copia intacta para poder comparar antes/después
df = df_original.copy()
df.head(10)

In [ ]:
# ANÁLISIS EXPLORATORIO INICIAL (antes de modificar nada)
print("Dimensiones (filas, columnas):", df.shape)
print("\nTipos de datos:")
print(df.dtypes)
print("\nValores nulos por columna:")
print(df.isna().sum())
print("\nFilas duplicadas exactas:", df.duplicated().sum())
print("order_id repetidos:", df["order_id"].duplicated().sum())
display(df.describe())

In [ ]:
# Valores de cada columna categórica (dropna=False para ver también los nulos)
for col in ["item_name", "item_type", "transaction_type", "received_by", "time_of_sale"]:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False))
    print()

In [ ]:
# Formato de las fechas: se observan dos estilos mezclados en la misma columna
es_barra = df["date"].str.contains("/")
print("Fechas con formato tipo 8/23/2022 (barras):", es_barra.sum())
print("Fechas con formato tipo 07-03-2022 (guiones):", (~es_barra).sum())
print("Ejemplos con barras :", df.loc[es_barra, "date"].head(3).tolist())
print("Ejemplos con guiones:", df.loc[~es_barra, "date"].head(3).tolist())

# ¿Las fechas con guiones son día-mes-año o mes-día-año? Ambos números son <= 12, así que a simple vista es ambiguo.
# Prueba: el rango de fechas de las ventas con barras (sin ambigüedad, pues el día llega hasta 31) es la referencia.
ref = pd.to_datetime(df.loc[es_barra, "date"], format="%m/%d/%Y")
guion = df.loc[~es_barra, "date"]
mes_dia = pd.to_datetime(guion, format="%m-%d-%Y")
dia_mes = pd.to_datetime(guion, format="%d-%m-%Y")
print(f"\nRango de referencia (con barras)        : {ref.min().date()} a {ref.max().date()}")
print(f"Guiones leídos como MES-DÍA-AÑO         : {mes_dia.min().date()} a {mes_dia.max().date()}")
print(f"Guiones leídos como DÍA-MES-AÑO         : {dia_mes.min().date()} a {dia_mes.max().date()}")

In [ ]:
# Consistencia interna: monto = precio unitario x cantidad ?
print("Filas donde precio x cantidad no coincide con el monto:",
      (df["item_price"] * df["quantity"] != df["transaction_amount"]).sum())
# Cada producto, ¿tiene un único precio y un único tipo?
display(df.groupby("item_name").agg(precios_distintos=("item_price", "nunique"),
                                    tipos_distintos=("item_type", "nunique")))
print("Rango de cantidad:", df["quantity"].min(), "a", df["quantity"].max())
print("Rango de precio  :", df["item_price"].min(), "a", df["item_price"].max())

### Problemas de calidad identificados antes de modificar

| # | Problema | Cómo se detectó | Magnitud |
|---|---|---|---|
| 1 | **Nulos en `transaction_type`** (medio de pago) | `isna().sum()` | 107 de 1.000 filas (10,7 %) |
| 2 | **Fechas en dos formatos mezclados** (`M/D/AAAA` y `MM-DD-AAAA`) | Revisión visual + conteo por presencia de `/` o `-` | 597 con barras y 403 con guiones |
| 3 | **Textos en inglés** (productos, tipos, turnos, medios de pago) | `value_counts()` | Todas las columnas de texto |
| 4 | **Columna `received_by`** ("Mr." / "Mrs.") sin significado claro para el negocio | `value_counts()` | 1.000 filas |
| 5 | **Moneda no especificada** | Los precios (20 a 60) no indican divisa | Toda la tabla |

Verificaciones que **no** encontraron problemas (y que se dejan documentadas): 0 filas duplicadas, `order_id` único, sin cantidades ni precios negativos o fuera de rango, y `precio × cantidad = monto` en las 1.000 filas.

## 2. Limpieza de datos

Cada decisión (qué se corrigió, qué se eliminó, qué se conservó) queda justificada en una celda de texto.

### Decisión 1 — Fechas: unificar formatos leyendo los guiones como MES-DÍA-AÑO

- **Qué encontramos:** la columna `date` mezcla `8/23/2022` (mes/día/año) con `07-03-2022`. En las de guiones, el día y el mes son siempre ≤ 12, por lo que a simple vista no se sabe si es día-mes o mes-día.
- **Cómo lo detectamos:** las fechas con barras no son ambiguas (llegan hasta el día 31) y cubren de abril 2022 a marzo 2023. Al leer las de guiones como DÍA-MES-AÑO las ventas se extienden de enero de **2022** a diciembre de **2023**, fuera del período del resto de los datos. Leídas como MES-DÍA-AÑO caen exactamente en el mismo período. Por eso el formato correcto es mes-día-año.
- **Decisión:** convertir todo a un único tipo `datetime` interpretando cada formato por separado. Leer todo con una sola regla (o con `dayfirst`) habría movido ventas al mes equivocado y dañado el reporte mensual.

In [ ]:
es_barra = df["date"].str.contains("/")
fecha_barra = pd.to_datetime(df["date"].where(es_barra), format="%m/%d/%Y", errors="coerce")
fecha_guion = pd.to_datetime(df["date"].where(~es_barra), format="%m-%d-%Y", errors="coerce")
df["fecha"] = fecha_barra.fillna(fecha_guion)

print("Fechas sin convertir:", df["fecha"].isna().sum())
print("Rango final:", df["fecha"].min().date(), "a", df["fecha"].max().date())

### Decisión 2 — Nulos en medio de pago: etiquetar como "No registrado"

- **Qué encontramos:** 107 ventas (10,7 %) sin medio de pago. Están repartidas de forma pareja entre turnos y productos, así que no siguen un patrón (parecen simple falta de registro).
- **Alternativas descartadas:**
  - *Eliminar las 107 filas*: se perdería el 10,7 % de las ventas y el monto total del negocio quedaría subestimado.
  - *Imputar con la moda (Efectivo)*: se inventaría información y se inflaría el efectivo.
- **Decisión:** conservar las filas y etiquetarlas como **"No registrado"**. Es la opción más transparente: los porcentajes muestran cuánto no se sabe. Las demás columnas de esas ventas sí son válidas y se siguen usando en el resto de reportes.

In [ ]:
print("Nulos antes:", df["transaction_type"].isna().sum())
df["transaction_type"] = df["transaction_type"].fillna("No registrado")
print("Nulos después:", df["transaction_type"].isna().sum())

### Decisión 3 — Duplicados: verificar y conservar todo

- **Cómo lo detectamos:** `df.duplicated()` (filas idénticas) y `order_id.duplicated()` (mismo pedido cargado dos veces con datos distintos).
- **Resultado:** 0 duplicados en ambos casos.
- **Decisión:** no se elimina nada. Se deja la verificación con código como evidencia reproducible de la calidad del dataset.

In [ ]:
print("Duplicados exactos:", df.duplicated().sum())
print("order_id repetidos:", df["order_id"].duplicated().sum())

### Decisión 4 — Traducir al español (los reportes deben salir en español)

- **Turnos:** Morning → Mañana, Afternoon → Tarde, **Evening → Atardecer**, **Night → Noche**, Midnight → Medianoche. Se guardan como categoría **ordenada** para que siempre salgan en el orden del día: Mañana, Tarde, Atardecer, Noche, Medianoche.
- **Medios de pago:** Cash → Efectivo, Online → En línea (más "No registrado").
- **Tipos de producto:** Fastfood → Comida rápida, Beverages → Bebidas.
- **Productos con traducción directa:** Cold coffee → Café frío, Sugarcane juice → Jugo de caña de azúcar, Sandwich → Sándwich.
- **Productos sin traducción adecuada:** *Aalopuri*, *Vadapav*, *Panipuri* y *Frankie* son platos típicos de la India (aperitivos y sándwiches callejeros) sin equivalente en español. **Se mantienen con su nombre original**, porque traducirlos con una descripción larga o un plato "parecido" alteraría lo que realmente se vende.
- **Nombres de columnas:** también se pasan al español para que los reportes sean claros.

### Decisión 5 — Columna `received_by`: eliminar

Solo contiene "Mr." o "Mrs." repartido casi 50/50 y no aclara quién es (¿el cliente?, ¿el empleado?). Ninguno de los reportes pedidos la necesita, y una interpretación equivocada podría llevar a conclusiones falsas. Se elimina del dataset limpio (sigue disponible en `df_original` si se necesitara).

### Decisión 6 — Moneda: dólares (US$)

El dataset no indica la moneda. Se **asume que los montos ya están en dólares** y se muestran con el prefijo "US$". **No se aplicó ninguna conversión**, porque habría que inventar un tipo de cambio.

In [ ]:
TURNOS = {"Morning": "Mañana", "Afternoon": "Tarde", "Evening": "Atardecer", "Night": "Noche", "Midnight": "Medianoche"}
ORDEN_TURNOS = ["Mañana", "Tarde", "Atardecer", "Noche", "Medianoche"]
PAGOS = {"Cash": "Efectivo", "Online": "En línea", "No registrado": "No registrado"}
ORDEN_PAGO = ["Efectivo", "En línea", "No registrado"]
TIPOS = {"Fastfood": "Comida rápida", "Beverages": "Bebidas"}
PRODUCTOS = {"Cold coffee": "Café frío", "Sugarcane juice": "Jugo de caña de azúcar", "Sandwich": "Sándwich",
             "Aalopuri": "Aalopuri", "Vadapav": "Vadapav", "Panipuri": "Panipuri", "Frankie": "Frankie"}
MESES = {1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril", 5: "Mayo", 6: "Junio", 7: "Julio",
         8: "Agosto", 9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"}
DIAS = {0: "Lunes", 1: "Martes", 2: "Miércoles", 3: "Jueves", 4: "Viernes", 5: "Sábado", 6: "Domingo"}
ORDEN_DIAS = list(DIAS.values())

limpio = pd.DataFrame({
    "id_orden": df["order_id"],
    "fecha": df["fecha"],
    "producto": df["item_name"].map(PRODUCTOS),
    "tipo_producto": df["item_type"].map(TIPOS),
    "precio_unitario": df["item_price"],
    "cantidad": df["quantity"],
    "monto": df["transaction_amount"],
    "medio_pago": df["transaction_type"].map(PAGOS),
    "turno": pd.Categorical(df["time_of_sale"].map(TURNOS), categories=ORDEN_TURNOS, ordered=True),
})
limpio["periodo"] = limpio["fecha"].dt.to_period("M")
limpio["dia_semana"] = pd.Categorical(limpio["fecha"].dt.dayofweek.map(DIAS), categories=ORDEN_DIAS, ordered=True)

# Si algún valor no estaba en los diccionarios de traducción, aparecería como nulo:
print("Nulos tras traducir:", limpio[["producto", "tipo_producto", "medio_pago", "turno"]].isna().sum().sum())
df = limpio
df.head()

### Verificación final de la limpieza

Se comprueba con código que cada paso tuvo el efecto esperado.

In [ ]:
assert df.shape[0] == df_original.shape[0], "Se perdieron filas"
assert df.drop(columns=["periodo", "dia_semana"]).isna().sum().sum() == 0, "Quedan nulos"
assert df["fecha"].dtype == "datetime64[ns]" or str(df["fecha"].dtype).startswith("datetime64"), "Fecha sin convertir"
assert (df["precio_unitario"] * df["cantidad"] == df["monto"]).all(), "Monto inconsistente"
assert set(df["medio_pago"]) == set(ORDEN_PAGO)
assert list(df["turno"].cat.categories) == ORDEN_TURNOS
assert df["id_orden"].is_unique

print("Filas conservadas      :", df.shape[0], "de", df_original.shape[0])
print("Nulos restantes        : 0")
print("Tipo de 'fecha'        :", df["fecha"].dtype)
print("Productos (en español) :", sorted(df["producto"].unique()))
print("Tipos de producto      :", sorted(df["tipo_producto"].unique()))
print("Medios de pago         :", sorted(df["medio_pago"].unique()))
print("Turnos (orden lógico)  :", list(df["turno"].cat.categories))
print("\nOK: todas las verificaciones pasaron.")

In [ ]:
# Exportar el dataset limpio (utf-8-sig para que Excel muestre bien las tildes)
df.drop(columns=["periodo", "dia_semana"]).to_csv("balaji_ventas_limpio.csv", index=False, encoding="utf-8-sig")
try:
    from google.colab import files
    files.download("balaji_ventas_limpio.csv")
except ImportError:
    print("Archivo guardado como balaji_ventas_limpio.csv")

In [ ]:
# Funciones para mostrar montos y porcentajes de forma legible
def usd(x):
    return f"US${x:,.0f}"

def usd2(x):
    return f"US${x:,.2f}"

def pct(x):
    return f"{x:.1f}%"

## 3. Reportes obligatorios

> Nota sobre las cifras: los montos están en dólares (US$), según el supuesto explicado en la Decisión 6. Cada fila del dataset es una venta de un solo producto.

### Reporte 1 — Ventas por medio de pago (efectivo vs en línea)

In [ ]:
r1 = (df.groupby("medio_pago")
        .agg(ventas=("id_orden", "count"), monto=("monto", "sum"))
        .reindex(ORDEN_PAGO))
r1["pct_ventas"] = r1["ventas"] / r1["ventas"].sum() * 100
r1["pct_monto"] = r1["monto"] / r1["monto"].sum() * 100
r1.loc["Total"] = [r1["ventas"].sum(), r1["monto"].sum(), 100.0, 100.0]

vista1 = pd.DataFrame({
    "Cantidad de ventas": r1["ventas"].astype(int),
    "% de las ventas": r1["pct_ventas"].map(pct),
    "Monto total": r1["monto"].map(usd),
    "% del monto": r1["pct_monto"].map(pct),
})
vista1.index.name = "Medio de pago"
display(vista1)

# Solo sobre las ventas donde sí se conoce el medio de pago (efectivo + en línea)
reg = r1.loc[["Efectivo", "En línea"]]
print("Porcentajes considerando solo las ventas con medio de pago registrado:")
for medio in reg.index:
    print(f"  {medio:9s}: {reg.loc[medio, 'ventas'] / reg['ventas'].sum() * 100:.1f}% de las ventas | "
          f"{reg.loc[medio, 'monto'] / reg['monto'].sum() * 100:.1f}% del monto")

ax = r1.loc[ORDEN_PAGO, "monto"].plot(kind="bar", color=["#2a9d8f", "#264653", "#adb5bd"], figsize=(6, 4))
ax.set_title("Monto vendido por medio de pago")
ax.set_xlabel("Medio de pago")
ax.set_ylabel("Monto (US$)")
plt.xticks(rotation=0)
plt.show()

### Reporte 2 — Productos por turno (unidades vendidas)

Productos elegidos: **Café frío, Sándwich y Panipuri**. Son productos distintos entre sí (bebida, comida más cara y comida económica) y muestran patrones distintos a lo largo del día.

In [ ]:
PRODUCTOS_R2 = ["Café frío", "Sándwich", "Panipuri"]
r2 = (df[df["producto"].isin(PRODUCTOS_R2)]
        .pivot_table(index="producto", columns="turno", values="cantidad", aggfunc="sum", observed=False)
        .reindex(PRODUCTOS_R2))
r2["Total"] = r2.sum(axis=1)
r2.index.name = "Producto"
r2.columns.name = "Turno (unidades vendidas)"
display(r2.astype(int))

ax = r2.drop(columns="Total").T.plot(kind="bar", figsize=(8, 4.5))
ax.set_title("Unidades vendidas por turno")
ax.set_xlabel("Turno")
ax.set_ylabel("Unidades vendidas")
ax.legend(title="Producto")
plt.xticks(rotation=0)
plt.show()

### Reporte 3 — Tipo de producto por medio de pago

In [ ]:
cols = ORDEN_PAGO + ["Total"]

# Cantidad de ventas
r3_v = pd.crosstab(df["tipo_producto"], df["medio_pago"], margins=True, margins_name="Total")[cols]
r3_v.index.name = "Tipo de producto"
r3_v.columns.name = "Cantidad de ventas"
display(r3_v)

# Monto vendido
r3_m = df.pivot_table(index="tipo_producto", columns="medio_pago", values="monto",
                      aggfunc="sum", margins=True, margins_name="Total")[cols]
r3_m.index.name = "Tipo de producto"
r3_m.columns.name = "Monto vendido"
display(r3_m.map(usd) if hasattr(r3_m, "map") else r3_m.applymap(usd))

# Cómo se reparte el monto de cada tipo de producto entre medios de pago (cada fila suma 100 %)
r3_p = r3_m[ORDEN_PAGO].div(r3_m["Total"], axis=0) * 100
r3_p.index.name = "Tipo de producto"
r3_p.columns.name = "% del monto de cada tipo"
display(r3_p.map(pct) if hasattr(r3_p, "map") else r3_p.applymap(pct))

### Reporte 4 — Producto más vendido de cada mes (por unidades y por ingreso)

Se muestran dos rankings porque no siempre coinciden: un plato barato puede vender muchas unidades y aun así aportar poco dinero.

In [ ]:
g = (df.groupby(["periodo", "producto"])
       .agg(unidades=("cantidad", "sum"), ingreso=("monto", "sum"))
       .reset_index())

filas = []
for periodo, sub in g.groupby("periodo"):
    top_u = sub.loc[sub["unidades"].idxmax()]
    top_i = sub.loc[sub["ingreso"].idxmax()]
    empates = (sub["unidades"] == sub["unidades"].max()).sum() + (sub["ingreso"] == sub["ingreso"].max()).sum() - 2
    filas.append({
        "Mes": f"{MESES[periodo.month]} {periodo.year}",
        "Más vendido (por unidades)": top_u["producto"],
        "Unidades": int(top_u["unidades"]),
        "Más vendido (por ingreso)": top_i["producto"],
        "Ingreso": usd(top_i["ingreso"]),
        "Coinciden": "Sí" if top_u["producto"] == top_i["producto"] else "No",
    })
    if empates:
        print(f"Aviso: hay empate en {periodo}")
r4 = pd.DataFrame(filas).set_index("Mes")
display(r4)
print("Meses donde el producto más vendido por unidades y por ingreso coincide:", (r4["Coinciden"] == "Sí").sum(), "de", len(r4))

## 4. Reportes adicionales

### Reporte A — Tamaño del pedido y su aporte al ingreso

**Pregunta de negocio:** ¿Conviene ofrecer descuentos o combos a quienes compran en cantidad, dado cuánto dinero aportan los pedidos grandes?

**Cómo se construye:** cada venta se clasifica según las unidades que incluye (Pequeño: 1 a 5, Mediano: 6 a 10, Grande: 11 a 15) y se compara cuántas ventas hay en cada grupo contra cuánto dinero aportan.

In [ ]:
etiquetas = ["Pequeño (1 a 5 unidades)", "Mediano (6 a 10 unidades)", "Grande (11 a 15 unidades)"]
df["tamano_pedido"] = pd.cut(df["cantidad"], bins=[0, 5, 10, 15], labels=etiquetas)
assert df["tamano_pedido"].notna().all()

ra = (df.groupby("tamano_pedido", observed=False)
        .agg(ventas=("id_orden", "count"), monto=("monto", "sum"), ticket=("monto", "mean")))
ra["pct_ventas"] = ra["ventas"] / ra["ventas"].sum() * 100
ra["pct_monto"] = ra["monto"] / ra["monto"].sum() * 100

vista_a = pd.DataFrame({
    "Cantidad de ventas": ra["ventas"],
    "% de las ventas": ra["pct_ventas"].map(pct),
    "Monto total": ra["monto"].map(usd),
    "% del monto": ra["pct_monto"].map(pct),
    "Monto promedio por venta": ra["ticket"].map(usd2),
})
vista_a.index.name = "Tamaño del pedido"
display(vista_a)

ax = ra[["pct_ventas", "pct_monto"]].rename(columns={"pct_ventas": "% de las ventas", "pct_monto": "% del dinero"}).plot(
    kind="bar", figsize=(8, 4.5), color=["#adb5bd", "#2a9d8f"])
ax.set_title("Los pedidos grandes: pocas ventas, mucho dinero")
ax.set_xlabel("Tamaño del pedido")
ax.set_ylabel("Porcentaje del total")
plt.xticks(rotation=0)
plt.show()

**Lectura:** los pedidos grandes representan alrededor de un tercio de las ventas pero más de la mitad del dinero. **Recomendación:** un descuento por volumen o combos de 10 unidades o más podría animar a más clientes a comprar en cantidad.

### Reporte B — Ventas por día de la semana

**Pregunta de negocio:** ¿Qué días de la semana conviene reforzar con más personal e inventario, y cuáles necesitan promociones para atraer clientes?

**Cómo se construye:** se calcula el día de la semana de cada fecha y se compara el número de ventas y el dinero que entra cada día.

In [ ]:
rb = (df.groupby("dia_semana", observed=False)
        .agg(ventas=("id_orden", "count"), monto=("monto", "sum"), ticket=("monto", "mean")))
rb["pct_monto"] = rb["monto"] / rb["monto"].sum() * 100

vista_b = pd.DataFrame({
    "Cantidad de ventas": rb["ventas"],
    "Monto total": rb["monto"].map(usd),
    "% del monto": rb["pct_monto"].map(pct),
    "Monto promedio por venta": rb["ticket"].map(usd2),
})
vista_b.index.name = "Día de la semana"
display(vista_b)

mejor, peor = rb["monto"].idxmax(), rb["monto"].idxmin()
print(f"Día con más ingreso: {mejor} ({usd(rb.loc[mejor, 'monto'])})")
print(f"Día con menos ingreso: {peor} ({usd(rb.loc[peor, 'monto'])})")

ax = rb["monto"].plot(kind="bar", figsize=(8, 4.5), color="#264653")
ax.set_title("Dinero vendido por día de la semana")
ax.set_xlabel("Día de la semana")
ax.set_ylabel("Monto (US$)")
plt.xticks(rotation=0)
plt.show()

## 5. Indicadores clave de desempeño (KPI)

A diferencia de un reporte (una tabla), un KPI es **un único valor con interpretación de negocio**.

### KPI 1 — Ticket promedio por venta

- **Qué mide:** cuánto dinero deja en promedio cada venta (monto total dividido entre el número de ventas).
- **Decisión que permite:** si el dueño quiere aumentarlo, puede armar combos o sugerir un producto adicional al momento de cobrar; cualquier mejora en este número se traduce directamente en más ingreso sin necesitar más clientes.

In [ ]:
ticket_promedio = df["monto"].mean()
print(f"KPI 1 - Ticket promedio por venta: {usd2(ticket_promedio)}")
print(f"        (calculado sobre {len(df)} ventas y {usd(df['monto'].sum())} en total)")

### KPI 2 — Crecimiento del ingreso trimestral

- **Qué mide:** cuánto cambió el dinero vendido entre el **primer trimestre** de los datos (abril a junio de 2022) y el **último** (enero a marzo de 2023).
- **Decisión que permite:** si el negocio crece de forma sostenida, se justifica reforzar personal e inventario y planificar más capacidad; si cayera, sería una señal para revisar precios, promociones o la competencia.

In [ ]:
primero = df[(df["fecha"] >= "2022-04-01") & (df["fecha"] <= "2022-06-30")]["monto"].sum()
ultimo = df[(df["fecha"] >= "2023-01-01") & (df["fecha"] <= "2023-03-31")]["monto"].sum()
crecimiento = (ultimo - primero) / primero * 100

print(f"Ingreso abril-junio 2022 : {usd(primero)}")
print(f"Ingreso enero-marzo 2023 : {usd(ultimo)}")
print(f"KPI 2 - Crecimiento del ingreso trimestral: {crecimiento:+.1f}%")

mensual = df.groupby("periodo")["monto"].sum()
mensual.index = [f"{MESES[p.month][:3]} {p.year}" for p in mensual.index]
ax = mensual.plot(kind="line", marker="o", figsize=(9, 4.5), color="#2a9d8f")
ax.set_title("Dinero vendido por mes")
ax.set_xlabel("Mes")
ax.set_ylabel("Monto (US$)")
plt.xticks(rotation=45)
plt.show()

## 6. Hallazgos para el dueño del restaurante

1. **Los pedidos grandes son los que sostienen el negocio.** Apenas un tercio de las ventas (las de más de 10 unidades) trae más de la mitad del dinero (55 %). Cada una deja en promedio US$439, frente a US$92 de un pedido pequeño. *Sugerencia:* ofrecer descuentos o combos por comprar en cantidad para atraer más pedidos de este tipo.

2. **El negocio está creciendo.** Entre abril-junio de 2022 y enero-marzo de 2023 el dinero vendido subió 29 % (de US$61,575 a US$79,655). *Sugerencia:* con esta tendencia, conviene planificar más personal e inventario y no quedarse corto de producto.

3. **Vender muchas unidades no es lo mismo que ganar más.** Aalopuri, Vadapav y Panipuri (los platos de US$20) suman el 40 % de las unidades vendidas, pero solo el 24 % del dinero. En cambio, Sándwich, Frankie y Café frío aportan el 65 % del dinero. *Sugerencia:* destacar en el menú y en la caja los productos de mayor valor, y usar los económicos como gancho o para acompañar.

4. **Algunos días y horarios rinden más.** El domingo es el día que más dinero deja (US$43,970) y el viernes y el martes los más flojos (alrededor de US$36,000). Además, el Sándwich se vende mucho más por la noche que por la tarde (292 contra 164 unidades), mientras que el Café frío tiene su mejor momento en la tarde. *Sugerencia:* reforzar personal los domingos, preparar más Sándwich para la noche y más Café frío para la tarde, y pensar promociones para martes y viernes.

5. **Casi 1 de cada 9 ventas no tiene registrado cómo pagó el cliente** (107 de 1.000, el 10,7 %). Entre las que sí se registraron, el efectivo sigue siendo un poco más usado que el pago en línea (53 % contra 47 %). *Sugerencia:* pedir al personal que anote siempre el medio de pago; sin ese dato no se puede saber con certeza cuánto conviene invertir en pagos digitales.

## 7. Consideraciones finales

*(Opinión del grupo sobre el logro del objetivo y el desarrollo de la asignación — completar.)*